In [2]:
#imports
import re
import os
import glob
import time
from datetime import datetime

#class definition

class syn_det:
    #path for file caontaining fingerprints
    fp_file_path = "./fingerprint.txt"
    #name for top module
    main_rtl_path = "fp_top"
    #name prefix for sub modules
    sub_module_prefix = "gen_number"

    # width map for tcp/ip fields
    width_map = {
        "ip_id": "16",
        "ip_tos": "8",
        "ip_src": "32",
        "ip_dst": "32",
        "tcp_sport": "16",
        "tcp_dport": "16",
        "tcp_seq": "32",
        "tcp_ack": "32",
        "tcp_window": "16",
        "tcp_urgptr": "16"
    }

    # set to store unique fields in fingerprints
    unique_fields = set()

    # maintain count of fields per fingerprint
    line_field_counts = []

    # list to store fields per fingerprint
    line_fields = []

    # maintain count of total fp
    total_fp = []

    #regexes used in the class
    
    #find dots in the string and replace with underscore
    dots_regex = r'\b(ip|tcp)\.(\w+)'
    dots_replacement   = r'\1_\2'

    # solves the funtion in fingerprint
    fl2b_regex = r'(?i)fl2b\((\w+)\)'
    fl2b_replacement   = r'\1[16-1:0]'

    #find variable with no width
    empty_width_regex = r"\b(tcp_\w+|ip_\w+)\b(?!\s*\[\d+-\d+:\d+\])"

    #tcp/ip field regex
    field_pattern_regex = r'\b(?:tcp|ip)_[^\]]+\]'

    #top module regex
    top_module_regex = r"(tcp_|ip_)(\w+)(\[\d+[-:\d]*\])"

    # constructor
    def __init__(self):
        self.width_regex = re.compile(self.empty_width_regex)
        self.field_regex = re.compile(self.field_pattern_regex)
        self.top_regex = re.compile(self.top_module_regex)
        pass

    # cleans up extra files
    def clean_up(self, i="all"):
        # Define the files to be deleted based on the input
        if i == "txt":
            files_to_delete = glob.glob("*.txt")
        elif i == "sv":
            files_to_delete = glob.glob("*.sv")
        elif i == "all":
            files_to_delete = glob.glob("*.txt") + glob.glob("*.sv")
        else:
            print("Invalid input. Please use 'txt', 'sv', or 'all'.")
            return

        # Remove files, ensuring "fingerprint.txt" is never deleted
        for file_path in files_to_delete:
            if os.path.basename(file_path) == "fingerprint.txt":
                continue  # Skip the fingerprint.txt file
            try:
                os.remove(file_path)
                print(f"Deleted: {file_path}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")

    def replace_dots(self, fp):
        out_str = re.sub(self.dots_regex, self.dots_replacement, fp)
        return out_str

    def solve_fl2b(self, fp):
        out_str = re.sub(self.fl2b_regex, self.fl2b_replacement, fp)
        return out_str

    def add_width(self, fp):
        matches = self.width_regex.findall(fp)
        for match in matches:
            if match in self.width_map:
                fp = fp.replace(match, f"{match}[{self.width_map[match]}-1:0]")
        return fp

    def extract_featrues(self, fp_list):
        self.line_field_counts = []
        self.line_fields = []
        for fp in fp_list:
            fields = self.field_regex.findall(fp)
            self.unique_fields.update(fields)
            self.line_field_counts.append(len(fields))
            self.line_fields.append(fields)
        self.total_fp = len(self.line_fields)

    def print_info(self):
        print(f"\n\nTotal Fingerprints found = {self.total_fp}\n")
        print(f"The total number of unique tcp/ip fields is {len(self.unique_fields)}\n")
        print(f"The following list contains all the unique fields:\n")
        temp = list(self.unique_fields)
        for item in self.unique_fields:
            print(f"{item}")
        print(f"\nList of Fields per fingerprint and fields counts are shown below:\n")
        for i in range(self.total_fp):
            print(f"Fields: {self.line_fields[i]}\nCount: {self.line_field_counts[i]}\n")

    def make_submodules(self, fp_list):
        # Get the current system username
        user_name = os.getlogin()
        # Get the current date and time
        current_time = datetime.now()
        # Format date as YYYY-MM-DD
        system_date = current_time.strftime("%Y-%m-%d")
        # Format time as HH:MM:SS
        system_time = current_time.strftime("%H:%M:%S")

        for loop_var in range(self.total_fp):
            submod_name = self.sub_module_prefix+'_'+str(loop_var)+".sv"
            with open(submod_name, 'w') as out:
                fp = fp_list[loop_var].strip()
                out.writelines(f"// Generated by {user_name} on {system_date} at {system_time}\n")
                out.writelines(f"// Auto-Generated sub-module for \n // {fp} \n")
                out.writelines(f"\n module {self.sub_module_prefix}_{loop_var}( \n\n")
                out.writelines(f"  input \t bit \t clk,\n")
                out.writelines(f"  input \t bit \t valid,\n")
                for i in range(self.line_field_counts[loop_var]):
                    items = self.line_fields[loop_var][i]
                    start_index = items.find('[')
                    end_index = items.find(']') + 1
                    var_width = items[start_index:end_index]
                    var_name = items[:start_index]
                    out.writelines(f"  input \t bit \t {var_width} \t {var_name},\n")
                out.writelines(f"\n  output \t logic \t fp_detected \n\n ); \n\n")
                
                for i in range(self.line_field_counts[loop_var]):
                   items = self.line_fields[loop_var][i]
                   start_index = items.find('[')
                   end_index = items.find(']') + 1
                   var_width = items[start_index:end_index]
                   var_name = items[:start_index]
                   out.writelines(f"  bit \t {var_width} \t {var_name}_f;")
                   out.writelines(f"\n  always @ (posedge clk) \n    {var_name}_f <= {var_name};\n\n")
                out.writelines(f"  bit \t valid_f;\n")
                out.writelines(f"  always @ (posedge clk) \n    valid_f <= valid;\n\n")
                temp_fp = fp.replace("[","_f[")
                out.writelines(f"  always @ (posedge clk) begin \n    if(valid_f) \n      fp_detected <= {temp_fp};")
                out.writelines(f" \n    else\n      fp_detected <= 0;\n  end \n\n endmodule \n")


    def make_top(self):
        sv_files = glob.glob("*.sv")
        module_list = [module_name[:-3] for module_name in sv_files]
        outfile = self.main_rtl_path+".sv"
        with open(outfile,'w') as out:
            out.writelines(f"// Auto-Generated top module for the syn-det generator \n\n")
            out.writelines(f"module {self.main_rtl_path} (\n\tinput\tbit\tclk,\n\tinput\tbit\trst, \n \tinput\tbit\tvalid,\n")
            for line in self.unique_fields:
                match = self.top_regex.match(line)
                if match:
                    prefix, name, brackets = match.groups()
                    out.writelines(f"\tinput \tbit\t{brackets}\t{prefix}{name},\n")
            for i in range(self.total_fp):
                out.writelines(f"\toutput\tlogic\t[16-1:0]\tfp_num{i},\n")
            out.writelines(f"\toutput\tlogic\tfp_det\n);\n")

            for xx in range(self.total_fp):
                out.writelines(f"// Instantiation tempelate for module {module_list[xx]}\n")
                out.writelines(f"\tlogic fp_detected_{xx};\n")
                out.writelines(f"\t{module_list[xx]} inst_{module_list[xx]} (\n")
                out.writelines(f"\t\t.clk(clk),\n")
                out.writelines(f"\t\t.valid(valid),\n")
                for i in range(self.line_field_counts[xx]):
                    items = self.line_fields[xx][i]
                    start_index = items.find('[')
                    end_index = items.find(']') + 1
                    var_name = items[:start_index]
                    out.writelines(f"\t\t.{var_name}({var_name}),\n")
                out.writelines(f"\t\t.fp_detected(fp_detected_{xx})\n")
                out.writelines(f"\t); \n\n")
            
            for i in range(self.total_fp):
                out.writelines(f"\talways@(posedge clk) begin\n\t\tif(fp_detected_{i} == 1'b1) begin\n\t\t\tfp_num{i} <= fp_num{i} + 1; \n\t\tend\n\tend\n")

            fp_det_list = [f"fp_detected_{i}\n\t\t\t\t" for i in range(self.total_fp)]
            fp_det_str = " || ".join(fp_det_list)
            out.writelines(f"\talways@(posedge clk) begin\n\t\tfp_det <= {fp_det_str};\n\tend\nendmodule")

    def make_instantiation(self):
        in_file = self.main_rtl_path+".sv"
        io_vars = []
        io_list = []
        with open(in_file,'r') as file: 
            for line in file:
                stripped_line = line.strip()
                if stripped_line.startswith(("input", "output")):
                    parts = stripped_line.split()
                    if parts[0] in ["input", "output"]:
                        parts = parts[1:]
                    if parts[0] == "bit":
                        parts[0] = "logic"
                    if parts[0] == "logic":
                        n_parts = parts[2:]
                    processed_line = " ".join(parts)
                    n_processed_line = " ".join(n_parts)
                    if processed_line.endswith(","):
                        processed_line = processed_line[:-1] + ";"
                    else:
                        processed_line += ";"
                    if n_processed_line.endswith(","):
                        n_processed_line = n_processed_line[:-1]
                    io_list.append(processed_line)
                    if n_processed_line != "":
                        io_vars.append(n_processed_line)

        outfile = "instantiation_tempelate.txt"
        with open(outfile,'w') as out:
            for i in range(len(io_list)):
                out.writelines(f"\t{io_list[i]}\n")
            out.writelines(f"// Instantiation tempelate for top module \n")
            out.writelines(f"\t{self.main_rtl_path} inst_{self.main_rtl_path} (\n\t\t.clk(clk),\n\t\t.rst(rst)\n")
            for i in range(len(io_vars)):
                out.writelines(f"\t\t.{io_vars[i]}({io_vars[i]}),\n")
            out.writelines(f"\t\t.fp_det(fp_det)\n\t);\n")

    def generate(self):
        fps = []
        self.clean_up()
        with open(self.fp_file_path,"r") as f:
            orig_fp = f.readlines()
        for line in orig_fp:
            line = self.replace_dots(line)
            line = self.solve_fl2b(line)
            line = self.add_width(line)
            fps.append(line)
        self.extract_featrues(fps)
        self.make_submodules(fps)
        self.make_top()
        self.make_instantiation()
        self.print_info()
        print(f"!!! Generation Succesfull !!!\n")

    def post_gen_cleaup(self):
        filename="instantiation_tempelate.txt"
         #remove duplication from var declaration
        pattern = re.compile(r'^(\s*logic\s*\[\s*(\d+)\s*-\s*1\s*:\s*0\s*\]\s*)(\w+)(\s*;.*)$')
        with open(filename, "r") as f:
            lines = f.readlines()
        best = {}  # var -> (max_width, line_index)
        for i, line in enumerate(lines):
            m = pattern.match(line)
            if not m:
                continue
            width = int(m.group(2))
            var = m.group(3)
            if var not in best or width > best[var][0]:
                best[var] = (width, i)
        keep = {idx for _, idx in best.values()}
        with open(filename, "w") as f:
            for i, line in enumerate(lines):
                m = pattern.match(line)
                if not m:
                    f.write(line)
                elif i in keep:
                    f.write(line)

        #remove duplication from actual instantiation
        pattern = re.compile(r'^\s*\.(\w+)\(\1\)\s*,?;?\s*$')
        with open(filename, "r") as f:
            lines = f.readlines()
        seen = set()
        with open(filename, "w") as f:
            for line in lines:
                m = pattern.match(line)
                if not m:
                    f.write(line)
                    continue
                var = m.group(1)
                if var not in seen:
                    seen.add(var)
                    f.write(line)

        #remove duplication from top_level file
        filename = self.main_rtl_path+".sv"
        pattern = re.compile(r'^(\s*input\s+bit\s+\[\s*(\d+)\s*-\s*1\s*:\s*0\s*\]\s*)(\w+)(\s*,.*)$')
        with open(filename, "r") as f:
            lines = f.readlines()
        best = {}  # var -> (max_width, line_index)
        for i, line in enumerate(lines):
            m = pattern.match(line)
            if not m:
                continue
            width = int(m.group(2))
            var = m.group(3)
            if var not in best or width > best[var][0]:
                best[var] = (width, i)
        keep = {idx for _, idx in best.values()}
        with open(filename, "w") as f:
            for i, line in enumerate(lines):
                m = pattern.match(line)
                if not m or i in keep:
                    f.write(line)

    def generate_filelist(self):
        directory="."
        output="filelist.sv"
        sub_files = sorted(
            f for f in os.listdir(directory)
            if f.startswith(self.sub_module_prefix) and f.endswith(".sv")
        )
        main_file = next(
            (f for f in os.listdir(directory)
             if f == f"{self.main_rtl_path}.sv" or (f.startswith(self.main_rtl_path) and f.endswith(".sv"))),
            None
        )
        with open(os.path.join(directory, output), "w") as f:
            for name in sub_files:
                f.write(f'`include "./{name}"\n')
            if main_file:
                f.write(f'`include "./{main_file}"\n')


start_time = time.perf_counter()
generator = syn_det()
generator.generate()
generator.post_gen_cleaup()
generator.generate_filelist()
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

Deleted: instantiation_tempelate.txt
Deleted: filelist.sv
Deleted: fp_top.sv
Deleted: gen_number_0.sv
Deleted: gen_number_1.sv
Deleted: gen_number_10.sv
Deleted: gen_number_11.sv
Deleted: gen_number_12.sv
Deleted: gen_number_13.sv
Deleted: gen_number_14.sv
Deleted: gen_number_15.sv
Deleted: gen_number_16.sv
Deleted: gen_number_2.sv
Deleted: gen_number_3.sv
Deleted: gen_number_4.sv
Deleted: gen_number_5.sv
Deleted: gen_number_6.sv
Deleted: gen_number_7.sv
Deleted: gen_number_8.sv
Deleted: gen_number_9.sv


Total Fingerprints found = 17

The total number of unique tcp/ip fields is 9

The following list contains all the unique fields:

tcp_seq[16-1:0]
tcp_dport[16-1:0]
tcp_window[16-1:0]
tcp_seq[32-1:0]
tcp_ack[32-1:0]
tcp_sport[16-1:0]
ip_id[16-1:0]
ip_dst[32-1:0]
ip_dst[16-1:0]

List of Fields per fingerprint and fields counts are shown below:

Fields: ['ip_dst[32-1:0]', 'tcp_seq[32-1:0]']
Count: 2

Fields: ['tcp_window[16-1:0]']
Count: 1

Fields: ['tcp_window[16-1:0]']
Count: 1

Fields